# Day 5 — Filtering, Sorting & Null Handling

**What you will learn:**
- Filtering with multiple conditions using `&`, `|`, `~`
- `sample()` — tricky exam favourite
- `collect()` vs `take()` vs `show()` — when to use each
- `distinct()` vs `dropDuplicates()`
- `orderBy()` / `sort()` with `asc()` / `desc()`
- Null handling: `dropna()`, `fillna()`, `na.replace()`

**Exam relevance:** DataFrame API (30%) — filtering and null handling appear in almost every practice test.

In [ ]:
from pyspark.sql.functions import col

data = [
    (1,  "Alice",   "RO", 95000,  "Engineering"),
    (2,  "Bob",     "RO", 72000,  "Marketing"),
    (3,  "Carol",   "UK", 110000, "Engineering"),
    (4,  "Dave",    "DE", 58000,  None),
    (5,  "Eve",     "RO", 81000,  "Marketing"),
    (6,  "Frank",   "DE", 88000,  "Engineering"),
    (7,  "George",  "UK", 72000,  "Marketing"),   # duplicate salary with Bob
    (8,  "Helen",   "RO", None,   "Sales"),
    (9,  "Ivan",    "DE", 88000,  None),           # duplicate salary with Frank
    (10, "Julia",   "UK", 95000,  "Engineering"),  # duplicate salary with Alice
]
df = spark.createDataFrame(data, ["id", "name", "country", "salary", "dept"])
df.show()

## 1. filter() / where() with Multiple Conditions

> **Exam tip:** Use `&` (AND), `|` (OR), `~` (NOT) — not Python's `and`/`or`/`not`.  
> Always wrap each condition in parentheses when combining.

`filter()` and `where()` are identical — both are valid.

In [ ]:
# AND — both conditions must be true
df.filter((col("country") == "RO") & (col("salary") > 75000)).show()

# OR — at least one condition true
df.filter((col("country") == "UK") | (col("salary") > 100000)).show()

# NOT — negate a condition
df.filter(~col("country").isin("DE", "UK")).show()

# where() is identical to filter()
df.where(col("dept") == "Engineering").show()

## 2. sample() — Know This for the Exam

`sample(withReplacement, fraction, seed)`

| Parameter | Type | Meaning |
|---|---|---|
| `withReplacement` | bool | `True` = same row can appear multiple times |
| `fraction` | float 0–1 | **Approximate** fraction — NOT guaranteed exact count |
| `seed` | int (optional) | For reproducibility |

> **Exam trap:** `sample()` returns *approximately* `fraction * total_rows` — not exactly.  
> `take(n)` returns *exactly* n rows, but always the first n (no randomness).

In [ ]:
# ~50% of rows, no replacement (unique rows only), reproducible
sample_no_replace = df.sample(False, 0.5, seed=42)
print(f"~50% sample (no replacement): {sample_no_replace.count()} rows")

# ~50% of rows WITH replacement (duplicates possible)
sample_with_replace = df.sample(True, 0.5, seed=42)
print(f"~50% sample (with replacement): {sample_with_replace.count()} rows")

# take(n) — returns EXACTLY n rows, but always the first n (no randomness)
first_three = df.take(3)
print(f"take(3): {len(first_three)} rows, type: {type(first_three)}")

## 3. collect() vs take() vs show()

| Method | Returns | Brings data to Driver | Use case |
|---|---|---|---|
| `show(n)` | None (prints) | Only n rows | Debug / inspect |
| `take(n)` | `List[Row]` | Exactly n rows | Small result sets |
| `first()` | `Row` | 1 row | Quick peek |
| `collect()` | `List[Row]` | **ALL rows** | Only on tiny DataFrames — OOM risk |

> **Exam tip:** `collect()` is an action that pulls all data to the Driver. Never use on large DataFrames.

In [ ]:
# show() — prints, returns None
df.show(3)

# take(n) — returns list of Row objects
rows = df.take(2)
print(type(rows), rows[0]["name"])

# first() — returns the first Row
first_row = df.first()
print(first_row["name"], first_row["salary"])

# collect() — brings everything to Driver (safe on small data)
all_rows = df.collect()
names = [r["name"] for r in all_rows]
print(names)

## 4. distinct() vs dropDuplicates()

| Method | Scope | Use case |
|---|---|---|
| `distinct()` | All columns | Remove rows where EVERY column is identical |
| `dropDuplicates([cols])` | Specified columns | Remove rows duplicate in specific columns only |

In [ ]:
# distinct() — removes rows where ALL columns are identical
print(f"Original: {df.count()} rows")
print(f"distinct: {df.distinct().count()} rows")  # same — no fully identical rows

# dropDuplicates on specific columns — keep first occurrence
deduped_salary = df.dropDuplicates(["salary"])
print(f"Unique salaries: {deduped_salary.count()} rows")
deduped_salary.select("name", "salary").show()

# dropDuplicates on multiple columns
deduped_country_dept = df.dropDuplicates(["country", "dept"])
deduped_country_dept.select("name", "country", "dept").show()

## 5. orderBy() / sort() and limit()

In [ ]:
from pyspark.sql.functions import asc, desc

# orderBy and sort are identical
df.orderBy(col("salary").desc()).show()

# Multiple sort columns
df.orderBy(["country", col("salary").desc()]).show()

# limit — equivalent to SQL LIMIT
df.orderBy(col("salary").desc()).limit(3).show()

## 6. Null Handling

> **Exam tip:** Nulls propagate in Spark — any expression involving null returns null.  
> `filter(col("x") != "value")` will silently drop rows where x is null.

In [ ]:
# Show rows with nulls
df.filter(col("salary").isNull() | col("dept").isNull()).show()

# dropna() — drop rows with ANY null value
df.dropna().show()

# dropna(how='all') — drop rows where ALL columns are null
df.dropna(how="all").show()

# dropna(subset=[...]) — drop rows with null in specific columns only
df.dropna(subset=["salary"]).show()

In [ ]:
# fillna() — replace nulls with a value
df.fillna(0, subset=["salary"]).show()           # fill numeric null
df.fillna("Unknown", subset=["dept"]).show()     # fill string null

# fillna with dict — different values per column
df.fillna({"salary": 0, "dept": "Unknown"}).show()

# na.replace() — replace specific values (not just nulls)
df.na.replace("RO", "Romania", subset=["country"]).show()

---

## Day 5 Checklist

- [ ] Used `&`, `|`, `~` for multi-condition filters (not `and`/`or`/`not`)
- [ ] Know `sample()` returns *approximately* the requested fraction
- [ ] Know the difference: `collect()` → all rows, `take(n)` → exactly n, `show()` → prints only
- [ ] Know when `distinct()` and `dropDuplicates([cols])` behave differently
- [ ] Used `orderBy()` with multiple columns and `asc()`/`desc()`
- [ ] Handled nulls with `dropna()`, `fillna()`, and `na.replace()`
- [ ] Know: nulls are silently excluded by inequality filters

**Next:** Day 6 — Spark SQL (createOrReplaceTempView, spark.sql)